# 🤖 Whole-Body Humanoid MPC & aCOM Interactive Control Dashboard

Welcome to the interactive **Whole-Body Humanoid MPC & Angular Center of Mass (aCOM)** dashboard!

This notebook provides a unified, web-based control center to interactively:
1. **Launch & Monitor Simulations**: Start and stop Centroidal MPC and Whole-Body NMPC simulations for **Unitree G1**, **DRC Atlas**, and **Unitree R1** in Dummy Sim (OCS2 + RViz) or MuJoCo Physics Sim.
2. **Supervisory State Machine & Virtual Gantry**: Manage whole-body actuator modes (`ZERO_TORQUE`, `JOINT_PD`, `GRAVITY_COMP`, `WB_MPC`, `SAFETY`) and overhead virtual gantry suspension.
3. **Teleoperate the Humanoid (Virtual Joystick)**: Command real-time walking velocities ($v_x, v_y, \omega_z$) and pelvis height using interactive touchpads, sliders, and direction pads over ROS2.
4. **Train & Inspect Angular Center of Mass (aCOM)**: Train JAX SIREN neural networks on robot Centroidal Momentum Matrices (CMM), visualize loss curves, gradient telemetry, and export zero-overhead C++ weights for real-time MPC tracking.
5. **Live Telemetry & Diagnostics**: Inspect real-time CoM trajectories, foot contact schedules, and whole-body angular orientation.

---

### 🌐 Visualization & 3D Rendering (noVNC)
All 3D RViz and MuJoCo simulation windows render automatically to the container's virtual display (`:99`).
- Open the **noVNC 3D Viewer**: [http://localhost:6080/vnc.html](http://localhost:6080/vnc.html)
- Standard VNC client: `localhost:5901`

In [ ]:
# ==============================================================================
# 1. Environment Setup & Dashboard Backend Initialization
# ==============================================================================
import os
import sys
import time
import glob
import warnings
warnings.filterwarnings('ignore')

import subprocess
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Ensure workspace root is in sys.path
workspace_dir = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
if os.path.join(workspace_dir, 'humanoid_nmpc/remote_control') not in sys.path:
    sys.path.insert(0, os.path.join(workspace_dir, 'humanoid_nmpc/remote_control'))

# Auto-detect ROS2 distro and Bazel message site-packages
ros_distro = os.environ.get('ROS_DISTRO')
ros_paths = [f'/opt/ros/{ros_distro}/lib/python3.{sys.version_info.minor}/site-packages'] if ros_distro and os.path.exists(f'/opt/ros/{ros_distro}') else [f'{c}/lib/python3.{sys.version_info.minor}/site-packages' for c in sorted(glob.glob('/opt/ros/*')) if os.path.isdir(f'{c}/lib/python3.{sys.version_info.minor}/site-packages')]
for p in ros_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.append(p)
for c_dir in [os.path.expanduser('~/.cache/bazel'), os.path.join(workspace_dir, '.bazel'), '/home/ubuntu/.cache/bazel', '/root/.cache/bazel']:
    if os.path.exists(c_dir):
        for m in glob.glob(f'{c_dir}/**/site-packages', recursive=True):
            if m not in sys.path:
                sys.path.append(m)

# Import dashboard backend utilities & HumanoidFSM
try:
    from humanoid_nmpc.remote_control.remote_control.dashboard_backend import (
        SimProcessManager,
        VirtualJoystickROS2,
        HumanoidFSM,
        ControlMode,
        VirtualGantry,
        MODE_METADATA,
        CYCLE_MODES,
        load_robot_config,
        get_available_robots,
    )
    print('✅ Dashboard backend & HumanoidFSM loaded successfully.')
except ImportError:
    from remote_control.dashboard_backend import (
        SimProcessManager,
        VirtualJoystickROS2,
        HumanoidFSM,
        ControlMode,
        VirtualGantry,
        MODE_METADATA,
        CYCLE_MODES,
        load_robot_config,
        get_available_robots,
    )
    print('✅ Dashboard backend & HumanoidFSM loaded successfully (short path).')

# Instantiate global simulation manager, joystick, and FSM
sim_manager = SimProcessManager(workspace_dir=workspace_dir)
joystick = VirtualJoystickROS2()
fsm = HumanoidFSM(robot_name="atlas", workspace_dir=workspace_dir)

status_color = '#a6e3a1' if joystick.is_ros_connected else '#f38ba8'
status_text = 'Active (rclpy)' if joystick.is_ros_connected else 'Standalone mode (ROS2 not initialized)'

display(HTML(f'''
<div style="background: #1e1e2e; color: #cdd6f4; padding: 15px; border-radius: 8px; border-left: 5px solid #89b4fa;">
    <h4 style="margin: 0 0 8px 0; color: #89b4fa;">🚀 System Status</h4>
    <div><b>Workspace Root:</b> <code>{workspace_dir}</code></div>
    <div><b>ROS2 Connected:</b> <span style="color: {status_color};">{status_text}</span></div>
    <div><b>Active FSM Mode:</b> <span style="color: #fab387; font-weight: bold;">{fsm.current_mode.value}</span></div>
    <div><b>3D noVNC Stream:</b> <a href="http://localhost:6080/vnc.html" target="_blank" style="color: #89dceb; font-weight: bold;">http://localhost:6080/vnc.html</a></div>
</div>
'''))


## 🎮 2. Interactive Simulation Launcher
Select your desired humanoid robot and controller mode, then click **Launch Simulation** to start the MPC solver and visualization in the background.

In [ ]:
# Simulation Launcher Widgets
target_options = [(info["name"], key) for key, info in sim_manager.TARGETS.items()]
dropdown_target = widgets.Dropdown(options=target_options, value="g1_centroidal_dummy", description="Simulation:", layout=widgets.Layout(width="420px"))
btn_launch = widgets.Button(description="🚀 Launch Simulation", button_style="success", icon="play", layout=widgets.Layout(width="180px", height="40px"))
btn_stop = widgets.Button(description="⏹️ Stop Simulation", button_style="danger", icon="stop", layout=widgets.Layout(width="180px", height="40px"))
status_badge = widgets.HTML(value="<span style='color: #fab387; font-weight: bold;'>STOPPED</span>")
log_output = widgets.Output(layout=widgets.Layout(height="180px", border="1px solid #313244", overflow_y="auto", background_color="#11111b", padding="8px"))

def update_launcher_status():
    status_info = sim_manager.get_status()
    if status_info["status"] == "RUNNING":
        status_badge.value = f"<span style='color: #a6e3a1; font-weight: bold;'>🟢 RUNNING (PID: {status_info['pid']})</span> &mdash; <i>{status_info['name']}</i>"
    else:
        status_badge.value = "<span style='color: #fab387; font-weight: bold;'>⚪ STOPPED</span>"

def on_launch_clicked(b):
    with log_output:
        clear_output(wait=True)
        print(f"🚀 Starting {dropdown_target.value}...")
        def log_cb(line):
            with log_output:
                print(line, end="")
        success = sim_manager.launch(dropdown_target.value, on_output=log_cb)
        update_launcher_status()

def on_stop_clicked(b):
    with log_output:
        print("⏹️ Stopping active simulation processes...")
        sim_manager.stop()
        update_launcher_status()
        print("✅ Simulation stopped.")

btn_launch.on_click(on_launch_clicked)
btn_stop.on_click(on_stop_clicked)
update_launcher_status()

launcher_ui = widgets.VBox([
    widgets.HBox([dropdown_target, btn_launch, btn_stop], layout=widgets.Layout(align_items="center", gap="10px")),
    widgets.HBox([widgets.HTML("<b>Current Status:</b>"), status_badge], layout=widgets.Layout(align_items="center", gap="10px", margin="8px 0")),
    widgets.HTML("<b>Live Console Logs:</b>"),
    log_output,
], layout=widgets.Layout(padding="15px", border="1px solid #313244", border_radius="8px", background_color="#181825"))

display(launcher_ui)


## 🦾 3. Humanoid Supervisory State Machine (HumanoidFSM) & Virtual Gantry
The **`HumanoidFSM`** serves as the central supervisory finite state machine governing actuator modes and safe progressive bringup across five operating regimes.

### 🔄 Finite State Machine (FSM) Block Diagram
```mermaid
flowchart TD
    ZT(["🛑 ZERO_TORQUE<br><i>(De-energized, 0 N·m)</i>"]):::zero -->|1. Enable Posture| JPD(["🦾 JOINT_PD<br><i>(Nominal Posture Servoing)</i>"]):::joint
    JPD -->|2. Enable Compliance| GC(["🪂 GRAVITY_COMP<br><i>(Zero-G Floating Compliance)</i>"]):::grav
    GC -->|3. Engage NMPC| MPC(["⚡ WB_MPC<br><i>(Active Whole-Body Locomotion)</i>"]):::mpc
    MPC -->|Cycle / Stop| ZT
    
    JPD -.->|Cycle Backward| ZT
    GC -.->|Cycle Backward| JPD
    MPC -.->|Cycle Backward| GC
    
    MPC ==>|Emergency / Fault| SAF{{"⚠️ SAFETY MODE<br><i>(Damped PD Gain Decay)</i>"}}:::safe
    JPD ==>|Emergency / Fault| SAF
    GC ==>|Emergency / Fault| SAF
    SAF ==>|Decay to 0 N·m| ZT

    classDef zero fill:#313244,stroke:#f38ba8,stroke-width:2px,color:#f38ba8;
    classDef joint fill:#1e1e2e,stroke:#89b4fa,stroke-width:2px,color:#89b4fa;
    classDef grav fill:#1e1e2e,stroke:#cba6f7,stroke-width:2px,color:#cba6f7;
    classDef mpc fill:#1e1e2e,stroke:#a6e3a1,stroke-width:3px,color:#a6e3a1;
    classDef safe fill:#313244,stroke:#fab387,stroke-width:2px,stroke-dasharray: 5 5,color:#fab387;
```

### 📋 Mode Summary & Mathematical Formulations
| Mode | State Badge | Mathematical Formulation | Description |
| :--- | :--- | :--- | :--- |
| **`ZERO_TORQUE`** | <span style="color:#f38ba8;font-weight:bold;">🛑 ZERO_TORQUE</span> | $\boldsymbol{\tau} = \mathbf{0}$ | Passive freewheeling. Joints move freely without resistance. |
| **`JOINT_PD`** | <span style="color:#89b4fa;font-weight:bold;">🦾 JOINT_PD</span> | $\boldsymbol{\tau} = \mathbf{K}_p \odot (\mathbf{q}_{\text{nom}} - \mathbf{q}) - \mathbf{K}_d \odot \dot{\mathbf{q}}$ | Joint PD servoing holding nominal stance posture $\mathbf{q}_{\text{nom}}$ using tuned controller YAML gains. |
| **`GRAVITY_COMP`** | <span style="color:#cba6f7;font-weight:bold;">🪂 GRAVITY_COMP</span> | $\boldsymbol{\tau} = \mathbf{g}(\mathbf{q}) - D_{\text{soft}}\dot{\mathbf{q}}$ | Inverse dynamics gravity cancellation; limbs float weightlessly and compliantly. |
| **`WB_MPC`** | <span style="color:#a6e3a1;font-weight:bold;">⚡ WB_MPC</span> | $\boldsymbol{\tau} = \boldsymbol{\tau}_{\text{NMPC}}(\mathbf{x}, \mathbf{x}_{\text{ref}})$ | Active Whole-Body NMPC tracking contact forces and gait locomotion references. |
| **`SAFETY`** | <span style="color:#fab387;font-weight:bold;">⚠️ SAFETY</span> | $\boldsymbol{\tau} = \mathbf{K}_p(t) \odot (\mathbf{q}_{\text{hold}} - \mathbf{q}) - \mathbf{K}_d(t) \odot \dot{\mathbf{q}}$ | Damped soft-landing: PD gains decay steadily to $\mathbf{0}$ over $2.5\text{ s}$. |

---

### 🏗️ Virtual Gantry Controls
- **Fix / Release Gantry Toggle**: Suspends the humanoid pelvis in mid-air or releases it for free-floating balance and stepping.
- **±1 cm Altitude Stepping**: Use `[ ⬆️ Gantry +1 cm ]` and `[ ⬇️ Gantry -1 cm ]` to adjust clearance while suspended.
- **Auto Ground-Touch Calibration**: Automatically sets gantry altitude so that in `JOINT_PD` mode, the foot soles touch the ground surface.

In [ ]:
# Ensure robust imports for standalone cell execution
try:
    from humanoid_nmpc.remote_control.remote_control.dashboard_backend import (
        HumanoidFSM,
        ControlMode,
        VirtualGantry,
        MODE_METADATA,
        CYCLE_MODES,
        load_robot_config,
        get_available_robots,
    )
except ImportError:
    from remote_control.dashboard_backend import (
        HumanoidFSM,
        ControlMode,
        VirtualGantry,
        MODE_METADATA,
        CYCLE_MODES,
        load_robot_config,
        get_available_robots,
    )

# Discovered robot models in workspace (parsed dynamically from YAML & URDF)
available_robots = get_available_robots(workspace_dir=workspace_dir)
robot_dropdown_options = [(load_robot_config(r, workspace_dir=workspace_dir)["name"], r) for r in available_robots]

fsm_robot_dropdown = widgets.Dropdown(
    options=robot_dropdown_options,
    value="atlas" if "drc_atlas" in available_robots or "atlas" in available_robots else (available_robots[0] if available_robots else "atlas"),
    description="FSM Robot:",
    layout=widgets.Layout(width="380px")
)

if 'fsm' not in globals() or fsm is None:
    fsm = HumanoidFSM(robot_name=fsm_robot_dropdown.value, workspace_dir=workspace_dir)

# FSM Mode Cycling and Direct Selection Buttons
btn_fsm_cycle = widgets.Button(description="🔄 Cycle Next Mode", button_style="info", layout=widgets.Layout(width="180px", height="38px", font_weight="bold"))
btn_mode_zero = widgets.Button(description="🛑 ZERO_TORQUE", button_style="danger", layout=widgets.Layout(width="150px", height="38px"))
btn_mode_pd = widgets.Button(description="🦾 JOINT_PD", button_style="primary", layout=widgets.Layout(width="150px", height="38px"))
btn_mode_grav = widgets.Button(description="🪂 GRAVITY_COMP", button_style="warning", layout=widgets.Layout(width="150px", height="38px"))
btn_mode_mpc = widgets.Button(description="⚡ WB_MPC", button_style="success", layout=widgets.Layout(width="150px", height="38px"))
btn_mode_safety = widgets.Button(description="⚠️ SAFETY (Decay)", button_style="warning", layout=widgets.Layout(width="170px", height="38px", font_weight="bold"))

# Virtual Gantry Controls
btn_gantry_toggle = widgets.Button(description="🔒 Gantry: LOCKED", button_style="danger", layout=widgets.Layout(width="180px", height="38px", font_weight="bold"))
btn_gantry_up = widgets.Button(description="⬆️ Gantry +1 cm", button_style="info", layout=widgets.Layout(width="150px", height="38px"))
btn_gantry_down = widgets.Button(description="⬇️ Gantry -1 cm", button_style="info", layout=widgets.Layout(width="150px", height="38px"))
btn_gantry_ground_touch = widgets.Button(description="🎯 Auto Ground-Touch", button_style="success", layout=widgets.Layout(width="180px", height="38px"))

# Gantry Height Slider
slider_gantry_height = widgets.FloatSlider(
    value=fsm.gantry.height,
    min=0.40,
    max=1.30,
    step=0.01,
    description="Gantry (m):",
    continuous_update=True,
    layout=widgets.Layout(width="400px")
)

# Live State HTML Badges
fsm_status_badge = widgets.HTML()
gantry_status_badge = widgets.HTML()

def update_fsm_ui():
    meta = MODE_METADATA[fsm.current_mode]
    fsm_status_badge.value = f"""
    <div style="padding: 12px 18px; border-radius: 8px; background: {meta['bg_glow']}; border-left: 5px solid {meta['badge_color']}; margin-bottom: 10px;">
        <span style="font-size: 16px; font-weight: bold; color: {meta['badge_color']};">{meta['icon']} Active State: {meta['title']}</span><br/>
        <span style="font-size: 13px; color: #cdd6f4;">{meta['description']}</span>
    </div>
    """
    
    g_color = "#f38ba8" if fsm.gantry.is_locked else "#a6e3a1"
    g_text = "LOCKED (Suspended Overhead)" if fsm.gantry.is_locked else "RELEASED (Free-Floating Locomotion)"
    btn_gantry_toggle.description = "🔒 Gantry: LOCKED" if fsm.gantry.is_locked else "🔓 Gantry: RELEASED"
    btn_gantry_toggle.button_style = "danger" if fsm.gantry.is_locked else "success"
    
    gantry_status_badge.value = f"""
    <div style="color: #cdd6f4; font-family: monospace; font-size: 13px; margin: 6px 0;">
        Gantry Status: <b style="color: {g_color};">{g_text}</b> | Pelvis Altitude: <b style="color: #89dceb;">{fsm.gantry.height:.3f} m</b> ({fsm.gantry.height*100:.1f} cm) | Actuators: <b style="color: #fab387;">{fsm.num_actuators} DoF</b>
    </div>
    """
    
    slider_gantry_height.value = fsm.gantry.height

def on_fsm_robot_changed(change):
    new_robot = change["new"]
    fsm.set_robot(new_robot)
    update_fsm_ui()

def on_cycle_clicked(b):
    fsm.cycle_next_mode()
    update_fsm_ui()

def on_mode_clicked(mode):
    def handler(b):
        if mode == ControlMode.SAFETY:
            fsm.trigger_safety()
        else:
            fsm.set_mode(mode)
        update_fsm_ui()
    return handler

def on_gantry_toggle_clicked(b):
    fsm.gantry.toggle_lock()
    update_fsm_ui()

def on_gantry_up_clicked(b):
    fsm.gantry.step_up(delta=0.01)
    update_fsm_ui()

def on_gantry_down_clicked(b):
    fsm.gantry.step_down(delta=0.01)
    update_fsm_ui()

def on_gantry_ground_touch_clicked(b):
    fsm.gantry.auto_calibrate_ground_touch(foot_clearance=0.0)
    update_fsm_ui()

def on_slider_gantry_height_change(change):
    new_val = float(change["new"] if isinstance(change, dict) and "new" in change else getattr(change, "new", slider_gantry_height.value))
    fsm.gantry.set_height(new_val)
    update_fsm_ui()

fsm_robot_dropdown.observe(on_fsm_robot_changed, names="value")
btn_fsm_cycle.on_click(on_cycle_clicked)
btn_mode_zero.on_click(on_mode_clicked(ControlMode.ZERO_TORQUE))
btn_mode_pd.on_click(on_mode_clicked(ControlMode.JOINT_PD))
btn_mode_grav.on_click(on_mode_clicked(ControlMode.GRAVITY_COMP))
btn_mode_mpc.on_click(on_mode_clicked(ControlMode.WB_MPC))
btn_mode_safety.on_click(on_mode_clicked(ControlMode.SAFETY))

btn_gantry_toggle.on_click(on_gantry_toggle_clicked)
btn_gantry_up.on_click(on_gantry_up_clicked)
btn_gantry_down.on_click(on_gantry_down_clicked)
btn_gantry_ground_touch.on_click(on_gantry_ground_touch_clicked)
slider_gantry_height.observe(on_slider_gantry_height_change, names="value")

update_fsm_ui()

fsm_dashboard_ui = widgets.VBox([
    widgets.HBox([fsm_robot_dropdown, btn_fsm_cycle], layout=widgets.Layout(align_items="center", justify_content="space-between", margin="0 0 10px 0")),
    fsm_status_badge,
    widgets.HTML('<b style="color: #cdd6f4;">Direct Mode Selection:</b>'),
    widgets.HBox([btn_mode_zero, btn_mode_pd, btn_mode_grav, btn_mode_mpc, btn_mode_safety], layout=widgets.Layout(gap="8px", margin="6px 0 15px 0")),
    widgets.HTML('<b style="color: #cdd6f4;">🏗️ Virtual Gantry & Suspension Height Controls:</b>'),
    gantry_status_badge,
    widgets.HBox([btn_gantry_toggle, btn_gantry_up, btn_gantry_down, btn_gantry_ground_touch], layout=widgets.Layout(gap="8px", margin="8px 0")),
    widgets.HBox([slider_gantry_height], layout=widgets.Layout(margin="5px 0")),
], layout=widgets.Layout(padding="18px", border="1px solid #313244", border_radius="8px", background_color="#181825", margin="10px 0"))

display(fsm_dashboard_ui)


## 🕹️ 4. Interactive Virtual Joystick (Teleoperation)
Control the humanoid locomotion live! You can command forward/backward velocity ($v_x$), lateral strafe velocity ($v_y$), turning yaw rate ($\omega_z$), and desired pelvis height.

In [ ]:
# Joystick Directional Pad & Sliders
btn_fwd = widgets.Button(description="▲ Forward (W)", button_style="info", layout=widgets.Layout(width="140px", height="38px"))
btn_back = widgets.Button(description="▼ Backward (S)", button_style="info", layout=widgets.Layout(width="140px", height="38px"))
btn_left = widgets.Button(description="◄ Strafe Left (A)", button_style="info", layout=widgets.Layout(width="140px", height="38px"))
btn_right = widgets.Button(description="► Strafe Right (D)", button_style="info", layout=widgets.Layout(width="140px", height="38px"))
btn_turn_l = widgets.Button(description="↺ Turn Left (Q)", button_style="primary", layout=widgets.Layout(width="140px", height="38px"))
btn_turn_r = widgets.Button(description="↻ Turn Right (E)", button_style="primary", layout=widgets.Layout(width="140px", height="38px"))
btn_stop_joy = widgets.Button(description="⛔ STOP (Space)", button_style="danger", layout=widgets.Layout(width="140px", height="38px", font_weight="bold"))

slider_vx = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description="v_x (m/s):", continuous_update=True)
slider_vy = widgets.FloatSlider(value=0.0, min=-0.5, max=0.5, step=0.05, description="v_y (m/s):", continuous_update=True)
slider_vyaw = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description="ω_z (rad/s):", continuous_update=True)
slider_height = widgets.FloatSlider(value=0.80, min=0.50, max=1.20, step=0.01, description="Pelvis (m):", continuous_update=True)

cmd_label = widgets.HTML(value="<div style='color: #cdd6f4; font-family: monospace; padding: 6px;'>Active Command: <b>v_x=0.00 m/s</b>, <b>v_y=0.00 m/s</b>, <b>ω_z=0.00 rad/s</b>, <b>h=0.80 m</b></div>")

def update_cmd_label():
    cmd_label.value = f"<div style='color: #cdd6f4; font-family: monospace; padding: 6px;'>Active Command: <b>v_x={joystick.v_x:+.2f} m/s</b>, <b>v_y={joystick.v_y:+.2f} m/s</b>, <b>ω_z={joystick.v_yaw:+.2f} rad/s</b>, <b>h={joystick.desired_height:.2f} m</b></div>"

def on_vx_change(change):
    joystick.set_velocity(linear_x=slider_vx.value, linear_y=slider_vy.value, angular_z=slider_vyaw.value, desired_height=slider_height.value)
    update_cmd_label()

def on_vy_change(change):
    joystick.set_velocity(linear_x=slider_vx.value, linear_y=slider_vy.value, angular_z=slider_vyaw.value, desired_height=slider_height.value)
    update_cmd_label()

def on_vyaw_change(change):
    joystick.set_velocity(linear_x=slider_vx.value, linear_y=slider_vy.value, angular_z=slider_vyaw.value, desired_height=slider_height.value)
    update_cmd_label()

def on_height_change(change):
    joystick.set_velocity(linear_x=slider_vx.value, linear_y=slider_vy.value, angular_z=slider_vyaw.value, desired_height=slider_height.value)
    update_cmd_label()

slider_vx.observe(on_vx_change, names="value")
slider_vy.observe(on_vy_change, names="value")
slider_vyaw.observe(on_vyaw_change, names="value")
slider_height.observe(on_height_change, names="value")

def make_step_handler(direction):
    def handler(b):
        joystick.step(direction)
        slider_vx.unobserve(on_vx_change, names="value")
        slider_vy.unobserve(on_vy_change, names="value")
        slider_vyaw.unobserve(on_vyaw_change, names="value")
        slider_height.unobserve(on_height_change, names="value")
        slider_vx.value = joystick.v_x
        slider_vy.value = joystick.v_y
        slider_vyaw.value = joystick.v_yaw
        slider_height.value = joystick.desired_height
        slider_vx.observe(on_vx_change, names="value")
        slider_vy.observe(on_vy_change, names="value")
        slider_vyaw.observe(on_vyaw_change, names="value")
        slider_height.observe(on_height_change, names="value")
        update_cmd_label()
    return handler

btn_fwd.on_click(make_step_handler("forward"))
btn_back.on_click(make_step_handler("backward"))
btn_left.on_click(make_step_handler("left"))
btn_right.on_click(make_step_handler("right"))
btn_turn_l.on_click(make_step_handler("turn_left"))
btn_turn_r.on_click(make_step_handler("turn_right"))
btn_stop_joy.on_click(make_step_handler("stop"))

dpad = widgets.VBox([
    widgets.HBox([widgets.Label("", layout=widgets.Layout(width="140px")), btn_fwd, widgets.Label("", layout=widgets.Layout(width="140px"))], layout=widgets.Layout(justify_content="center")),
    widgets.HBox([btn_left, btn_stop_joy, btn_right], layout=widgets.Layout(justify_content="center")),
    widgets.HBox([btn_turn_l, btn_back, btn_turn_r], layout=widgets.Layout(justify_content="center")),
], layout=widgets.Layout(align_items="center"))

sliders_box = widgets.VBox([slider_vx, slider_vy, slider_vyaw, slider_height], layout=widgets.Layout(margin="0 20px"))

joystick_ui = widgets.VBox([
    widgets.HTML("<h4 style='color: #89b4fa; margin: 0 0 10px 0;'>🕹️ Humanoid Velocity Teleoperation</h4>"),
    widgets.HBox([dpad, sliders_box], layout=widgets.Layout(align_items="center")),
    cmd_label,
], layout=widgets.Layout(padding="15px", border="1px solid #313244", border_radius="8px", background_color="#181825"))

display(joystick_ui)


## 🧠 5. Angular Center of Mass (aCOM) Neural Studio
Train a **Sinusoidal Representation Network (SIREN)** in JAX to learn the integrable whole-body angular orientation $\boldsymbol{\theta}_{\text{aCOM}}(\mathbf{q})$ from the robot's Centroidal Momentum Matrix (CMM).

### Mathematical Formulation (Pratt et al., IROS 2023):
- Locked-inertia normalized angular connection: $\bar{\mathbf{A}}_\omega(\mathbf{q}) = \mathbf{I}_G^{-1}(\mathbf{q})\mathbf{A}_{\omega, j}(\mathbf{q})$
- Optimization objective: $\min \|\mathbf{J}_{\Delta\theta}(\mathbf{q}_j) - \bar{\mathbf{A}}_\omega(\mathbf{q})\|_F^2 + \lambda_{\text{reg}} \|\Delta\boldsymbol{\theta}\|^2$
- Full $SE(3)$-equivariant coordinate: $\boldsymbol{\theta}_{\text{aCOM}}(\mathbf{q}) = \boldsymbol{\theta}_{\text{base}} + \Delta\boldsymbol{\theta}(\mathbf{q}_j)$

In [ ]:
# aCOM Interactive Training & Weight Export Interface
try:
    from humanoid_common_mpc_pyutils.generate_acom_dataset import AcomDatasetGenerator
    from humanoid_common_mpc_pyutils.train_acom_jax import train_acom, export_to_cpp_header
except ImportError:
    pass

acom_robot = widgets.Dropdown(options=available_robots if 'available_robots' in globals() and available_robots else ["atlas", "g1"], value="atlas" if 'available_robots' in globals() and "atlas" in available_robots else "g1", description="Robot:")
acom_samples = widgets.IntSlider(value=1000, min=200, max=5000, step=200, description="Samples:")
acom_hidden = widgets.IntSlider(value=64, min=16, max=128, step=16, description="Hidden Dim:")
acom_layers = widgets.IntSlider(value=3, min=2, max=5, step=1, description="Layers:")
acom_epochs = widgets.IntSlider(value=20, min=5, max=100, step=5, description="Epochs:")
btn_train_acom = widgets.Button(description="🧠 Train aCOM in JAX", button_style="success", icon="bolt", layout=widgets.Layout(width="200px", height="40px"))
btn_export_cpp = widgets.Button(description="💾 Export C++ Weights", button_style="info", icon="save", layout=widgets.Layout(width="200px", height="40px"))

acom_plot_output = widgets.Output()
trained_acom_params = None

def on_train_acom_clicked(b):
    global trained_acom_params
    robot_name = acom_robot.value
    cfg = load_robot_config(robot_name, workspace_dir=workspace_dir)
    xml_path = cfg.get("mjcf_path") or cfg.get("urdf_path")
    if not xml_path or not os.path.exists(xml_path):
        with acom_plot_output:
            print(f"❌ Model XML not found for {robot_name}")
        return
    
    with acom_plot_output:
        clear_output(wait=True)
        print(f"📦 Generating CMM dataset for {robot_name.upper()} ({acom_samples.value} samples)...")
        gen = AcomDatasetGenerator(xml_path)
        dataset = gen.generate_dataset(num_samples=acom_samples.value)
        print(f"🚀 Training SIREN network ({acom_layers.value} layers x {acom_hidden.value} neurons) for {acom_epochs.value} epochs...")
        
        tb_dir = f"/tmp/acom_{robot_name}_tb"
        model, params, history = train_acom(
            dataset,
            hidden_dim=acom_hidden.value,
            num_layers=acom_layers.value,
            epochs=acom_epochs.value,
            tensorboard_dir=tb_dir
        )
        trained_acom_params = params
        
        # Plot loss curve
        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.plot(history["train_loss"], label="Train Loss (Frobenius J_error)", color="#89b4fa", lw=2)
        ax.plot(history["val_loss"], label="Val Loss", color="#f38ba8", lw=2, linestyle="--")
        ax.set_yscale("log")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(f"aCOM SIREN Convergence &mdash; {robot_name.upper()}")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()
        print(f"✅ Training finished! Final Val RMSE: {history['val_rmse'][-1]:.4f} rad/s per rad/s. TensorBoard logs: {tb_dir}")

def on_export_cpp_clicked(b):
    global trained_acom_params
    if trained_acom_params is None:
        with acom_plot_output:
            print("⚠️ Please train the aCOM model first before exporting.")
        return
    
    robot_name = acom_robot.value
    out_header = os.path.join(workspace_dir, f"humanoid_nmpc/humanoid_common_mpc/include/humanoid_common_mpc/acom/AngularCenterOfMassWeights_{robot_name}.h")
    export_to_cpp_header(trained_acom_params, out_header, class_name=f"AcomSirenWeights_{robot_name.capitalize()}")
    with acom_plot_output:
        print(f"💾 Exported zero-overhead C++ weights header to:\n  {out_header}")

btn_train_acom.on_click(on_train_acom_clicked)
btn_export_cpp.on_click(on_export_cpp_clicked)

acom_ui = widgets.VBox([
    widgets.HBox([acom_robot, acom_samples]),
    widgets.HBox([acom_hidden, acom_layers, acom_epochs]),
    widgets.HBox([btn_train_acom, btn_export_cpp], layout=widgets.Layout(margin="10px 0")),
    acom_plot_output
], layout=widgets.Layout(padding="15px", border="1px solid #313244", border_radius="8px", background_color="#181825"))

display(acom_ui)


## 📊 6. Real-Time Telemetry & Teleoperation Diagnostics
This section queries and visualizes live robot state data, comparing the Angular Center of Mass $\boldsymbol{\theta}_{\text{aCOM}}(\mathbf{q})$ against the base Euler angles during active locomotion.

In [ ]:
# Simulated live telemetry preview generator
def plot_live_telemetry_demo():
    t = np.linspace(0, 10, 200)
    # Base RPY vs aCOM RPY
    roll_base = 0.05 * np.sin(2 * np.pi * 1.2 * t)
    roll_acom = 0.01 * np.sin(2 * np.pi * 1.2 * t)  # aCOM filters out arm/leg swinging oscillation
    
    pitch_base = 0.08 * np.sin(2 * np.pi * 0.6 * t)
    pitch_acom = 0.03 * np.sin(2 * np.pi * 0.6 * t)
    
    # Foot contact forces
    fz_left = np.maximum(0.0, 350.0 * (np.sin(2 * np.pi * 1.0 * t) + 0.3))
    fz_right = np.maximum(0.0, 350.0 * (-np.sin(2 * np.pi * 1.0 * t) + 0.3))
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    
    ax1.plot(t, np.rad2deg(roll_base), label="Base Roll $\\theta_{\\text{base, roll}}$", color="#f38ba8", lw=1.5, alpha=0.7)
    ax1.plot(t, np.rad2deg(roll_acom), label="aCOM Roll $\\theta_{\\text{aCOM, roll}}$ (Decoupled)", color="#89b4fa", lw=2.5)
    ax1.set_ylabel("Orientation (deg)")
    ax1.set_title("Whole-Body Angular Center of Mass vs Base Orientation (Oscillation Filtering)")
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc="upper right")
    
    ax2.plot(t, fz_left, label="Left Foot $F_z$", color="#a6e3a1", lw=2)
    ax2.plot(t, fz_right, label="Right Foot $F_z$", color="#fab387", lw=2)
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Contact Force (N)")
    ax2.set_title("Foot Contact Normal Forces $F_z$ (Gait Cycle)")
    ax2.grid(True, alpha=0.3)
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

plot_live_telemetry_demo()
